# Анализ новорегов

In [92]:
import requests
import json
import sys
import time
from pprint import pprint
from typing import Optional, List
from dataclasses import dataclass
from datetime import datetime


API_URL = "https://gql.twitch.tv/gql"
API_CLIENT_ID = "kd1unb4b3q4t58fwlpcbzcbnm76a8fp"
USER_DATA_REQUEST = """
query fetchUser($id: ID, $login: String, $first: Int = 100, $after: Cursor) {
  user(id: $id, login: $login, lookupType: ALL) {
    id
    login
    createdAt
    deletedAt
    description
    language
    settings {
        preferredLanguageTag
    }
    follows(first: $first, after: $after) {
        totalCount
        edges {
            cursor
            node {
                id
                login
            }
        }
    }
  }
}
"""


@dataclass
class ShortUserData:
    id: int
    login: str
    

@dataclass
class UserData:
    id: int
    login: str
    createdAt: datetime
    deletedAt: datetime
    description: str
    language: str
    preferredLanguageTag: str
    follows_count: int
    follows: List[ShortUserData]


def load_user(id: Optional[int] = None,
              login: Optional[str] = None,
              repeat_times: int = 10,
              repead_delay: float = 0.5) -> Optional[UserData]:
    session = requests.Session()

    for _ in range(repeat_times):
        response = session.post(
            url=API_URL,
            json={
                'query': USER_DATA_REQUEST,
                'variables': {
                    'id': None if id is None else str(id),
                    'login': login
                }
            },
            headers={
                "Client-ID": API_CLIENT_ID
            })

        if response.status_code != 200:
            print("[ERROR]", "Status code =", response.status_code)
            
            time.sleep(repead_delay)            
            continue

        data = json.loads(response.text)
        if 'errors' in data.keys() and len(data['errors']) > 0:
            for error in data['errors']:
                print("[ERROR]", error)

            time.sleep(repead_delay)            
            continue

        def json_to_short_user(data) -> ShortUserData:
            return ShortUserData(id=data['node']['id'], login=data['node']['login'])
        
        if data['data']['user'] is None:
            return None
        else:
            return UserData(id=int(data['data']['user']['id']),
                            login=data['data']['user']['login'],
                            createdAt=datetime.fromisoformat(data['data']['user']['createdAt']),
                            deletedAt=None if data['data']['user']['deletedAt'] is None else datetime.fromisoformat(data['data']['user']['deletedAt']),
                            description=data['data']['user']['description'],
                            language=data['data']['user']['language'],
                            preferredLanguageTag=data['data']['user']['settings']['preferredLanguageTag'],
                            follows_count=int(data['data']['user']['follows']['totalCount']),
                            follows=list(map(json_to_short_user, data['data']['user']['follows']['edges'])))

In [55]:
left = 1
right = 1_500_695_511

while right - left > 1:
    mid = (left + right) // 2
    user = load_user(id=mid)
    if user is None:
        right = mid
    else:
        left = mid

In [57]:
MAX_ID = left
print(MAX_ID)

1437642599


In [58]:
left = 1
right = MAX_ID

while right - left > 1:
    mid = (left + right) // 2
    user = load_user(id=mid)
    assert user is not None

    if user.createdAt.year < 2025 or user.createdAt.year == 2025 and user.createdAt.month < 3:
        left = mid
    else:
        right=mid

In [59]:
MIN_ID = right

In [60]:
print(MIN_ID, MAX_ID, MAX_ID - MIN_ID + 1)

1271699294 1437642599 165943306


In [102]:
from random import randint

user = load_user(randint(MIN_ID, MAX_ID))
user.language, user.preferredLanguageTag, user

('RU',
 None,
 UserData(id=1435192227, login='environmentalmiter9psrvj6', createdAt=datetime.datetime(2026, 1, 28, 18, 4, 47, 966814, tzinfo=datetime.timezone.utc), deletedAt=None, description=None, language='RU', preferredLanguageTag=None, follows_count=0, follows=[]))

In [106]:
# 1000 - minute
str(100_000_000 / 1000 / 60 / 24) + "days"

'69.44444444444444days'